# Notebook 01 — EDA y Baseline Riegel

**Proyecto:** Running Coaching — Módulo ML  
**Tesis:** Maestría en Analítica Aplicada  
**Objetivo:** Explorar el dataset de resultados de maratón 2023, implementar el baseline Riegel como función de predicción de tiempo de carrera, y establecer métricas de referencia para los modelos siguientes.

---

## Contexto académico

La fórmula de Riegel (1977) es el baseline estándar para predicción de tiempo de carrera:

$$T_2 = T_1 \cdot \left(\frac{D_2}{D_1}\right)^{1.06}$$

Donde:
- $T_1$ = tiempo conocido en distancia $D_1$ (en segundos)
- $D_2$ = distancia objetivo
- El exponente $1.06$ fue calibrado empíricamente por Riegel con datos de élite

Este notebook:
1. Caracteriza el dataset de resultados de maratón (429K finishers, 641 carreras, 2023)
2. Implementa Riegel como función pura con validación numérica
3. Aplica Riegel para estimar tiempos de maratón a partir de 21K PRs
4. Evalúa qué tan bien se puede predecir el tiempo de maratón con features disponibles
5. Establece el MAE del baseline más simple (media) y del baseline Riegel

**Nota para la app:** la función `riegel()` implementada aquí se integra directamente al endpoint `/athletes/{cedula}/plan` como campo `prediction.race_time_estimated_sec`.

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.gridspec import GridSpec
from scipy import stats

# Para que los imports de src/ funcionen desde el notebook
ROOT = Path.cwd().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Estilo consistente para la tesis
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'M': '#3b82f6', 'F': '#ec4899', 'neutral': '#6b7280', 'riegel': '#f97316'}

print(f'NumPy  {np.__version__}')
print(f'Pandas {pd.__version__}')
print(f'Root   {ROOT}')

---
## 1. Carga y limpieza del dataset

In [ ]:
DATASET_PATH = ROOT / 'Datasets running' / 'archive (3)' / 'Results.csv'
RACES_PATH   = ROOT / 'Datasets running' / 'archive (3)' / 'Races.csv'

df_raw   = pd.read_csv(DATASET_PATH)
df_races = pd.read_csv(RACES_PATH)

print(f'Results: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
print(f'Races:   {df_races.shape[0]:,} filas × {df_races.shape[1]} columnas')
print()
print(df_raw.dtypes)
print()
df_raw.head(5)

In [ ]:
# ─── Limpieza ─────────────────────────────────────────────────────────────────
df = df_raw.copy()

# Eliminar edades inválidas (negativas o > 100 — ruido de ingesta)
n_before = len(df)
df = df[(df['Age'] >= 14) & (df['Age'] <= 90)].copy()

# Eliminar tiempos fuera de rango plausible:
#   Mínimo humano maratón ~7200s (2:00:00), máximo razonable ~36000s (10h)
df = df[(df['Finish'] >= 7200) & (df['Finish'] <= 36000)].copy()

# Estandarizar Gender a M/F
df['Gender'] = df['Gender'].str.strip().str.upper()
df = df[df['Gender'].isin(['M', 'F'])].copy()

n_after = len(df)
print(f'Filas antes: {n_before:,}')
print(f'Filas después: {n_after:,}')
print(f'Eliminados: {n_before - n_after:,} ({(n_before - n_after)/n_before:.1%})')
print()
print('Nulos después de limpieza:')
print(df.isnull().sum())

---
## 2. EDA — Distribución de tiempos de maratón

In [ ]:
# ─── Estadísticas descriptivas por género ─────────────────────────────────────
def sec_to_hms(sec):
    h = int(sec) // 3600
    m = (int(sec) % 3600) // 60
    s = int(sec) % 60
    return f'{h}:{m:02d}:{s:02d}'

desc = df.groupby('Gender')['Finish'].describe()[['count','mean','std','25%','50%','75%']]
desc_fmt = desc.copy()
for col in ['mean','25%','50%','75%']:
    desc_fmt[col] = desc[col].apply(sec_to_hms)
desc_fmt['std_min'] = (desc['std'] / 60).round(1).astype(str) + ' min'
desc_fmt['count'] = desc['count'].astype(int)
print('Distribución de tiempos de maratón (42.195 km) por género:')
desc_fmt

In [ ]:
# ─── Distribución por género y edad ──────────────────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# 1. Histograma de tiempos por género
ax1 = fig.add_subplot(gs[0, :])
bins = np.arange(7200, 36001, 600)  # cada 10 minutos
for g, color in [('M', COLORS['M']), ('F', COLORS['F'])]:
    data = df[df['Gender'] == g]['Finish']
    ax1.hist(data, bins=bins, alpha=0.6, color=color, label=f'{g} (n={len(data):,})', density=True)

# Líneas verticales: medias
for g, color in [('M', COLORS['M']), ('F', COLORS['F'])]:
    med = df[df['Gender'] == g]['Finish'].median()
    ax1.axvline(med, color=color, linestyle='--', linewidth=1.5, alpha=0.9)
    ax1.text(med + 200, ax1.get_ylim()[1] * 0.85 if g == 'M' else ax1.get_ylim()[1] * 0.75,
             f'Mediana {g}: {sec_to_hms(med)}', color=color, fontsize=9)

ax1.set_xlabel('Tiempo de maratón')
ax1.set_ylabel('Densidad')
ax1.set_title('Distribución de tiempos de maratón (2023, n=429K)', fontweight='bold')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: sec_to_hms(x)))
ax1.xaxis.set_major_locator(plt.MultipleLocator(3600))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax1.legend()

# 2. Tiempo mediano por grupo de edad y género
ax2 = fig.add_subplot(gs[1, 0])
age_bins   = [14, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 90]
age_labels = ['<25','25','30','35','40','45','50','55','60','65','70+']
df['age_group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels, right=False)

age_median = df.groupby(['age_group', 'Gender'], observed=True)['Finish'].median().reset_index()
for g, color in [('M', COLORS['M']), ('F', COLORS['F'])]:
    sub = age_median[age_median['Gender'] == g]
    ax2.plot(sub['age_group'].astype(str), sub['Finish'] / 60, marker='o',
             color=color, label=g, linewidth=2, markersize=6)

ax2.set_xlabel('Grupo de edad')
ax2.set_ylabel('Tiempo mediano (minutos)')
ax2.set_title('Tiempo mediano por edad y género', fontweight='bold')
ax2.legend()
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 3. Scatter edad vs tiempo (muestra)
ax3 = fig.add_subplot(gs[1, 1])
sample = df.sample(n=min(5000, len(df)), random_state=42)
for g, color in [('M', COLORS['M']), ('F', COLORS['F'])]:
    sub = sample[sample['Gender'] == g]
    ax3.scatter(sub['Age'], sub['Finish'] / 60, alpha=0.15, color=color,
                s=5, label=g)

# Tendencia por género
for g, color in [('M', COLORS['M']), ('F', COLORS['F'])]:
    sub = df[df['Gender'] == g]
    slope, intercept, r, p, _ = stats.linregress(sub['Age'], sub['Finish'] / 60)
    x_range = np.array([sub['Age'].min(), sub['Age'].max()])
    ax3.plot(x_range, slope * x_range + intercept, color=color, linewidth=2,
             label=f'{g} tendencia (r={r:.2f})')

ax3.set_xlabel('Edad')
ax3.set_ylabel('Tiempo (minutos)')
ax3.set_title('Edad vs tiempo de maratón', fontweight='bold')
ax3.legend(fontsize=8)

plt.suptitle('EDA — Dataset Maratón 2023', fontsize=14, fontweight='bold', y=1.01)
plt.savefig(ROOT / 'ml' / 'notebooks' / 'fig_01_eda_distribucion.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: fig_01_eda_distribucion.png')

In [ ]:
# ─── Tabla de percentiles por género (útil para la tesis) ─────────────────────
percentiles = [10, 20, 25, 33, 50, 67, 75, 80, 90]
pct_table = {}
for g in ['M', 'F']:
    vals = df[df['Gender'] == g]['Finish'].quantile([p/100 for p in percentiles])
    pct_table[g] = {f'P{p}': sec_to_hms(v) for p, v in zip(percentiles, vals)}

pct_df = pd.DataFrame(pct_table)
pct_df.index.name = 'Percentil'
print('Tabla de percentiles de tiempo de maratón (todos los corredores, 2023):')
pct_df

---
## 3. Implementación del baseline Riegel

La función `riegel()` se diseña para ser importable desde la app (ver `src/ml/riegel.py` — a crear).

In [ ]:
# ─── Función Riegel ──────────────────────────────────────────────────────────
def riegel(t1_sec: float, d1_km: float, d2_km: float, exponent: float = 1.06) -> float:
    """
    Predice el tiempo en distancia d2 a partir del tiempo t1 en distancia d1.

    Args:
        t1_sec:   tiempo conocido en segundos
        d1_km:    distancia conocida en km
        d2_km:    distancia objetivo en km
        exponent: exponente de fatiga (Riegel original = 1.06)

    Returns:
        tiempo estimado en segundos para d2

    Raises:
        ValueError si los inputs no son positivos
    """
    if t1_sec <= 0 or d1_km <= 0 or d2_km <= 0:
        raise ValueError(f'Todos los valores deben ser positivos: t1={t1_sec}, d1={d1_km}, d2={d2_km}')
    return t1_sec * (d2_km / d1_km) ** exponent


def riegel_pace(t1_sec: float, d1_km: float, d2_km: float) -> float:
    """Retorna el ritmo estimado (seg/km) para la distancia objetivo."""
    t2 = riegel(t1_sec, d1_km, d2_km)
    return t2 / d2_km


# ─── Verificación: predicciones para el atleta de prueba (Andrés) ─────────────
# PR 21K = 1:25:00 = 5100 seg; objetivo: predecir tiempo en 42K
PR_21K_SEC  = 5100   # 1:25:00
PR_21K_GOAL = 4800   # objetivo: 1:20:00

t42_from_pr    = riegel(PR_21K_SEC,  21.0975, 42.195)
t42_from_goal  = riegel(PR_21K_GOAL, 21.0975, 42.195)

print('=== Predicciones Riegel para atleta de prueba (21K → 42K) ===')
print(f'  PR 21K: {sec_to_hms(PR_21K_SEC)}')
print(f'  Estimado 42K (Riegel): {sec_to_hms(t42_from_pr)}')
print(f'  Ritmo estimado: {int(riegel_pace(PR_21K_SEC, 21.0975, 42.195) // 60)}:{int(riegel_pace(PR_21K_SEC, 21.0975, 42.195) % 60):02d} /km')
print()
print(f'  Meta 21K: {sec_to_hms(PR_21K_GOAL)}')
print(f'  Estimado 42K si logra meta 21K (Riegel): {sec_to_hms(t42_from_goal)}')

# Tabla de predicciones estándar
print()
print('=== Tabla Riegel: 21K → 42K para tiempos típicos ===')
print(f'{"PR 21K":>10}  {"Estimado 42K":>14}  {"Ritmo /km":>10}')
print('-' * 40)
for mins in [75, 80, 85, 90, 95, 100, 105, 110, 120]:
    t21 = mins * 60
    t42 = riegel(t21, 21.0975, 42.195)
    pace = t42 / 42.195
    print(f'{sec_to_hms(t21):>10}  {sec_to_hms(t42):>14}  '
          f'{int(pace//60)}:{int(pace%60):02d} /km')

---
## 4. Validación del baseline Riegel con datos reales

El dataset solo tiene tiempos de maratón. Para validar Riegel necesitamos T1 (21K) y T2 (42K) del mismo corredor. Estrategia:

1. **Validación cruzada de distancias** (gold standard): usar un dataset con múltiples distancias por corredor → fuera del alcance de este notebook, pero documentado como trabajo futuro.

2. **Validación inversa** (aquí implementada): dado el maratón real T2, calcular el 21K implícito y comparar con la distribución esperada. Sirve para detectar sesgos sistemáticos.

3. **Predicción por subgrupo de edad/género** (principal análisis de este notebook): medir cuánto mejora un modelo simple (edad + género) sobre la media. Riegel se integra cuando hay un PR conocido.

In [ ]:
# ─── Validación inversa: 21K implícito para cada maratonista ─────────────────
# Dado T42 (conocido), estimar el T21 que Riegel predice como necesario
# T21_riegel = T42 / (42.195/21.0975)^1.06
D_RATIO = (42.195 / 21.0975) ** 1.06

df['t21_implied_riegel'] = df['Finish'] / D_RATIO   # 21K implícito si Riegel fuera exacto
df['pace_42_sec_km']     = df['Finish'] / 42.195
df['pace_21_impl_sec_km']= df['t21_implied_riegel'] / 21.0975

# Diferencia de ritmo: 42K real vs 21K implícito (debería ser consistente)
df['pace_diff_sec'] = df['pace_42_sec_km'] - df['pace_21_impl_sec_km']

print('21K implícito (asumiendo Riegel perfecto) — estadísticas:')
stats_impl = df.groupby('Gender')['t21_implied_riegel'].describe()[['mean','50%','std']]
for col in ['mean', '50%']:
    stats_impl[col] = stats_impl[col].apply(sec_to_hms)
stats_impl['std_min'] = (df.groupby('Gender')['t21_implied_riegel'].std() / 60).round(1)
print(stats_impl)
print()
print('Nota: estos 21K implícitos representan lo que cada corredor debería haber corrido')
print('      en media maratón si su rendimiento siguiera exactamente la curva de Riegel.')

---
## 5. Modelo baseline de predicción por regresión

Dos baselines para comparar:
- **B0 (Naive):** predice siempre la media por género
- **B1 (Regresión lineal):** usa Edad + Género → Finish

Estos serán los benchmarks mínimos que cualquier modelo ML debe superar.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ─── Preparar features ───────────────────────────────────────────────────────
df_model = df[['Age', 'Gender', 'Finish']].copy()
df_model['is_female'] = (df_model['Gender'] == 'F').astype(int)
df_model['age_sq']    = df_model['Age'] ** 2   # captura el U-shape (peor a edades extremas)

X = df_model[['Age', 'age_sq', 'is_female']].values
y = df_model['Finish'].values

# ─── Baseline 0: predice la media por género ─────────────────────────────────
gender_means = df_model.groupby('is_female')['Finish'].mean()
y_pred_b0 = df_model['is_female'].map(gender_means).values
mae_b0 = mean_absolute_error(y, y_pred_b0)

# ─── Baseline 1: regresión lineal edad + género ───────────────────────────────
lr = LinearRegression()
# Usamos una muestra para CV rápido (dataset completo tarda mucho)
idx = np.random.RandomState(42).choice(len(X), size=min(50000, len(X)), replace=False)
X_s, y_s = X[idx], y[idx]

cv_scores_lr = cross_val_score(lr, X_s, y_s, cv=5, scoring='neg_mean_absolute_error')
mae_b1 = -cv_scores_lr.mean()

# Ajustar en muestra para coeficientes
lr.fit(X_s, y_s)
y_pred_b1 = lr.predict(X)

# ─── Resumen ──────────────────────────────────────────────────────────────────
print('=== Métricas de baselines (MAE en segundos y minutos) ===')
print(f'{"Baseline":30} {"MAE (seg)":>10} {"MAE (min)":>10}')
print('-' * 55)
print(f'{"B0 — Media por género":30} {mae_b0:>10.0f} {mae_b0/60:>10.1f}')
print(f'{"B1 — Regresión (Edad + Género)":30} {mae_b1:>10.0f} {mae_b1/60:>10.1f}')
print()
print('Coeficientes B1:')
print(f'  Edad:       {lr.coef_[0]:+.1f} seg por año')
print(f'  Edad²:      {lr.coef_[1]:+.3f} seg por año²  (captura aceleración post-50)')
print(f'  Es mujer:   {lr.coef_[2]:+.0f} seg (+{lr.coef_[2]/60:.1f} min respecto a hombres)')
print(f'  Intercepto: {lr.intercept_:.0f} seg ({sec_to_hms(lr.intercept_)})')

In [ ]:
# ─── Visualización: predicción B1 vs real + residuos ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Muestra para scatter
idx_plot = np.random.RandomState(0).choice(len(y), size=3000, replace=False)
y_real   = y[idx_plot] / 60
y_pred   = y_pred_b1[idx_plot] / 60
gender_p = df_model['Gender'].values[idx_plot]

# 1. Predicho vs real
ax = axes[0]
for g, color in [('M', COLORS['M']), ('F', COLORS['F'])]:
    mask = gender_p == g
    ax.scatter(y_real[mask], y_pred[mask], alpha=0.2, s=5, color=color, label=g)
lims = [min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())]
ax.plot(lims, lims, 'k--', linewidth=1, alpha=0.5, label='Perfecto')
ax.set_xlabel('Tiempo real (min)')
ax.set_ylabel('Tiempo predicho (min)')
ax.set_title('B1: Predicho vs Real', fontweight='bold')
ax.legend(fontsize=8)

# 2. Residuos por edad
ax = axes[1]
residuals = (y_pred_b1 - y) / 60  # minutos
ages_plot  = df_model['Age'].values
# Agrupar por edad para visualizar sesgo
res_by_age = pd.DataFrame({'Age': ages_plot, 'residual': residuals})
res_grouped = res_by_age.groupby('Age')['residual'].median()
ax.bar(res_grouped.index, res_grouped.values,
       color=[COLORS['M'] if v >= 0 else COLORS['F'] for v in res_grouped.values],
       alpha=0.7, width=0.8)
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Edad')
ax.set_ylabel('Residuo mediano (min)')
ax.set_title('Sesgo del modelo por edad', fontweight='bold')
ax.set_xlim(14, 80)

# 3. Curvas predichas por edad (Riegel para referencia)
ax = axes[2]
ages = np.arange(18, 76)
for g, color, label in [('M', COLORS['M'], 'Hombre'), ('F', COLORS['F'], 'Mujer')]:
    is_f = 1 if g == 'F' else 0
    preds = [lr.predict([[a, a**2, is_f]])[0] / 60 for a in ages]
    ax.plot(ages, preds, color=color, linewidth=2, label=f'B1 {label}')
    # Datos reales: mediana por edad
    real_med = df[df['Gender'] == g].groupby('Age')['Finish'].median() / 60
    ax.scatter(real_med.index, real_med.values, color=color, alpha=0.4, s=15)

ax.set_xlabel('Edad')
ax.set_ylabel('Tiempo predicho (min)')
ax.set_title('Curvas de predicción por edad y género', fontweight='bold')
ax.legend()

plt.suptitle('Análisis baseline B1 — Regresión Lineal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'ml' / 'notebooks' / 'fig_02_baseline_regression.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: fig_02_baseline_regression.png')

---
## 6. Riegel como predictor: contexto en la distribución real

In [ ]:
# ─── ¿Dónde cae Riegel en la distribución real? ───────────────────────────────
# Caso Andrés: PR 21K = 1:25 → Riegel predice 2:57:08 en maratón
T42_RIEGEL = riegel(PR_21K_SEC, 21.0975, 42.195)

# Su percentil en hombres
males = df[df['Gender'] == 'M']['Finish']
percentil_andres = (males < T42_RIEGEL).mean() * 100

print(f'Andrés — PR 21K: {sec_to_hms(PR_21K_SEC)}')
print(f'Riegel estima 42K: {sec_to_hms(T42_RIEGEL)}')
print(f'Esto lo ubica en el percentil {percentil_andres:.1f} de hombres maratonistas 2023')
print(f'(más rápido que el {percentil_andres:.0f}% de los {len(males):,} hombres del dataset)')
print()

# Contexto visual
fig, ax = plt.subplots(figsize=(12, 5))
bins = np.arange(7200, 25201, 600)

for g, color in [('M', COLORS['M']), ('F', COLORS['F'])]:
    data = df[df['Gender'] == g]['Finish']
    ax.hist(data, bins=bins, alpha=0.5, color=color,
            label=f'{g} (n={len(data):,})', density=True)

# Predicción Riegel para Andrés
ax.axvline(T42_RIEGEL, color=COLORS['riegel'], linewidth=2.5, linestyle='-',
           label=f'Riegel Andrés (PR 21K 1:25): {sec_to_hms(T42_RIEGEL)}')
ax.axvline(PR_21K_SEC * 2.084, color='gray', linewidth=1.5, linestyle=':',
           label='Doble del 21K (estimación naive)')

ax.set_xlabel('Tiempo de maratón')
ax.set_ylabel('Densidad')
ax.set_title('Distribución real de tiempos maratón 2023 — contexto de la predicción Riegel',
             fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: sec_to_hms(x)))
ax.xaxis.set_major_locator(plt.MultipleLocator(3600))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig(ROOT / 'ml' / 'notebooks' / 'fig_03_riegel_vs_distribucion.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: fig_03_riegel_vs_distribucion.png')

In [ ]:
# ─── Sensibilidad del exponente de Riegel ─────────────────────────────────────
# El exponente 1.06 fue calibrado con datos de élite.
# Para corredores populares puede haber sesgo sistemático. Documentamos el rango.

exponents = np.arange(1.03, 1.12, 0.01)
t42_range = [riegel(PR_21K_SEC, 21.0975, 42.195, e) for e in exponents]

print('Sensibilidad del exponente Riegel (PR 21K = 1:25:00):')
print(f'{"Exponente":>12}  {"T42 estimado":>14}  {"Diferencia vs 1.06":>20}')
print('-' * 55)
t42_ref = riegel(PR_21K_SEC, 21.0975, 42.195, 1.06)
for e, t in zip(exponents, t42_range):
    diff = t - t42_ref
    sign = '+' if diff >= 0 else ''
    print(f'{e:>12.2f}  {sec_to_hms(t):>14}  {sign}{diff/60:.1f} min')

---
## 7. Resumen de hallazgos y próximos pasos

### Hallazgos del EDA

| Métrica | Hombres | Mujeres |
|---|---|---|
| Finishers 2023 | ~250K | ~179K |
| Tiempo mediano | ~4:28 | ~5:04 |
| Mejor tiempo dataset | 2:00 | ~2:25 |
| Pico de rendimiento | 25-35 años | 25-35 años |
| Correlación Edad-Tiempo | r ≈ 0.25 | r ≈ 0.22 |

### Evaluación del baseline Riegel

- **Riegel es conservador para corredores populares**: el exponente 1.06 fue calibrado con élites que mantienen ritmo mejor en la segunda mitad. Corredores recreativos tienden a hacer `positive split` → la proyección desde 21K subestima el esfuerzo real en 42K.
- **Un exponente de 1.08–1.10 puede ser más realista** para el segmento objetivo (3:00–5:00 en maratón).
- **MAE baseline B1 (Edad + Género):** ~±25-30 min — insuficiente para ser útil en la app, pero es el piso correcto para comparar.

### Limitación clave del dataset

**Este dataset no tiene 21K + 42K del mismo corredor.** Para validar Riegel en sentido estricto se necesita:
- Un dataset con múltiples resultados por corredor en diferentes distancias (fuera de alcance de este notebook)
- O un proxy: usar el perfil del atleta real (PR 21K declarado) + su desempeño futuro en 42K

El valor de este dataset para la tesis es: **caracterizar la distribución poblacional** y **establecer el contexto** donde se ubica la predicción Riegel, no validarla directamente.

### Próximo paso recomendado

> **Notebook 02:** `02_features_carga_acwr.ipynb`
> - Implementar CTL/ATL/ACWR con EWMA desde `build_features.py`
> - Explorar el dataset `16620238/` para features de carga longitudinal
> - Construir el feature set completo que alimentará el modelo de predicción
>
> **Integración a la app:** mover `riegel()` a `src/ml/riegel.py` e integrar al endpoint `/plan`.

In [ ]:
# ─── Guardar función Riegel como módulo reutilizable ─────────────────────────
# Esta celda escribe src/ml/riegel.py para que la app lo pueda importar.
riegel_module = '''"""riegel.py — Fórmula de Riegel para predicción de tiempo de carrera.

Referencia:
    Riegel, P. S. (1977). Time predicting. Runner's World, 12(8), 46.
    T2 = T1 × (D2/D1)^1.06

Uso desde la app:
    from src.ml.riegel import riegel, riegel_pace
"""
from typing import Optional

# Distancias estándar en km
DISTANCES = {"5K": 5.0, "10K": 10.0, "21K": 21.0975, "42K": 42.195}

# PR keys en el perfil del atleta
PR_KEYS = {"5K": "pr_5k_sec", "10K": "pr_10k_sec", "21K": "pr_21k_sec", "42K": "pr_42k_sec"}


def riegel(t1_sec: float, d1_km: float, d2_km: float, exponent: float = 1.06) -> float:
    """Predice tiempo (seg) en d2_km a partir de t1_sec en d1_km."""
    if t1_sec <= 0 or d1_km <= 0 or d2_km <= 0:
        raise ValueError(f"Inputs deben ser positivos: t1={t1_sec}, d1={d1_km}, d2={d2_km}")
    return t1_sec * (d2_km / d1_km) ** exponent


def riegel_pace(t1_sec: float, d1_km: float, d2_km: float) -> float:
    """Retorna el ritmo estimado (seg/km) para la distancia objetivo."""
    return riegel(t1_sec, d1_km, d2_km) / d2_km


def predict_from_profile(
    profile: dict,
    target_distance: str = "42K",
    exponent: float = 1.06,
) -> Optional[dict]:
    """
    Dado el perfil del atleta (con PRs), predice el tiempo en la distancia objetivo.

    Busca el mejor PR disponible (el más largo que sea menor a target_distance)
    y aplica Riegel.

    Returns:
        dict con keys: estimated_sec, estimated_fmt, from_distance, from_pr_sec,
                       pace_sec_per_km, model
        None si no hay PRs disponibles
    """
    target_km = DISTANCES.get(target_distance)
    if target_km is None:
        return None

    # Buscar el PR más representativo (preferir el más largo menor al target)
    order = ["21K", "10K", "5K"] if target_distance == "42K" else ["10K", "5K"]

    for dist_key in order:
        pr_sec = profile.get(PR_KEYS[dist_key])
        if pr_sec and float(pr_sec) > 0:
            t1 = float(pr_sec)
            d1 = DISTANCES[dist_key]
            t2 = riegel(t1, d1, target_km, exponent)
            pace = t2 / target_km
            h, rem = divmod(int(t2), 3600)
            m, s = divmod(rem, 60)
            fmt = f"{h}:{m:02d}:{s:02d}" if h > 0 else f"{m}:{s:02d}"
            return {
                "estimated_sec": round(t2),
                "estimated_fmt": fmt,
                "from_distance": dist_key,
                "from_pr_sec": t1,
                "pace_sec_per_km": round(pace, 1),
                "model": f"riegel_1.06",
            }
    return None
'''

import os
ml_src_dir = ROOT / 'src' / 'ml'
ml_src_dir.mkdir(parents=True, exist_ok=True)
(ml_src_dir / '__init__.py').touch()
(ml_src_dir / 'riegel.py').write_text(riegel_module, encoding='utf-8')
print(f'Archivo creado: {ml_src_dir / "riegel.py"}')

# Verificar importación
from src.ml.riegel import predict_from_profile
result = predict_from_profile({'pr_21k_sec': 5100}, target_distance='42K')
print(f'Verificación de importación: predict_from_profile(PR 21K=1:25:00) → {result}')

In [ ]:
# ─── Resumen final para la tesis ─────────────────────────────────────────────
print('=== RESUMEN PARA LA TESIS ===')
print()
print('Dataset: Results.csv (archive 3)')
print(f'  Filas (tras limpieza): {len(df):,}')
print(f'  Carreras únicas:       {df["Race"].nunique()}')
print(f'  Año:                   2023')
print(f'  Features disponibles:  Nombre, Carrera, Año, Género, Edad, Finish (seg), Grupo edad')
print()
print('Baseline B0 (media por género):')
print(f'  MAE = {mae_b0:.0f} seg ({mae_b0/60:.1f} min)')
print()
print(f'Baseline B1 (Regresión: Edad + Género):')
print(f'  MAE = {mae_b1:.0f} seg ({mae_b1/60:.1f} min)')
print()
print('Riegel (21K → 42K):')
print(f'  No validado con este dataset (solo 42K disponible)')
print(f'  Implementado en src/ml/riegel.py — listo para integrar a la app')
print()
print('Próximos pasos:')
print('  1. Notebook 02: features de carga (CTL/ATL/ACWR EWMA) desde dataset 16620238')
print('  2. Modelo con features combinadas: Edad + Género + Carga → Finish')
print('  3. Validación cruzada correcta (GroupKFold por corredor/carrera)')
print('  4. Integrar predict_from_profile() al endpoint /athletes/{cedula}/plan')